### RAG Test and Evaluation

In [1]:
import os
from pathlib import Path
os.chdir(path = Path(r"C:\Users\apaks\projects\YT-RAG"))

In [2]:
from src.yt_rag.components.data_loader import DataLoader
from src.yt_rag.components.embedding import EmbeddingManager
from src.yt_rag.components.vectorstore import FaissVectorStore, VectorStoreManager
from src.yt_rag.components.search import RAGSearch
import time
from dotenv import load_dotenv

load_dotenv()

c:\Users\apaks\projects\YT-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
# VectorStoreManager().reset()

In [4]:
url = "https://www.youtube.com/watch?v=5t1vTLU7s40"

In [5]:
query = "How does the RTX 50 Series use AI to process images differently than traditional rendering?"

In [6]:
def run_rag_pipeline(url, query):
    rag = RAGSearch(url = url)
    start = time.time()
    relevant_chunks = rag.search(query= query, top_k = 5)
    context = " ".join(relevant_chunks)
    response = rag.generate_response(context=context, query=query)
    timestamps = rag.get_video_timestamps()
    end = time.time()

    runtime_duration = end-start
    return {"answer": response, "relevant_chunks": relevant_chunks,"timestamps": timestamps, "runtime_duration": runtime_duration}

In [7]:
# print(result)
# print(timestamps)
# print(runtime_duration)

In [8]:
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "YouTube-RAG"

In [9]:
from langsmith import Client

# initiate langsmith client
client = Client()

In [10]:
# Define examples
examples = [
    {
        "inputs": {"question": "Why does Yann LeCun believe auto-regressive LLMs are limited for achieving human-level intelligence?"},
        "outputs": {"answer": "LLMs lack four essential characteristics: an understanding of the physical world, persistent memory, the ability to reason, and the ability to plan. They generate text token-by-token without the abstract, deliberate thought processes humans use before speaking."}
    },
    {
        "inputs": {"question": """What is a "Joint-Embedding Predictive Architecture" (JEPA) and how does it differ from generative models?"""},
        "outputs": {"answer": """Unlike generative models that spend resources attempting to reconstruct every detail or pixel of an input, JEPA learns abstract representations of data. It operates in an abstract space to predict future states without needing to generate the original, full-resolution input."""}
    },
    {
        "inputs": {"question": """Why does LeCun argue that open-sourcing AI is essential for democracy?"""},
        "outputs": {"answer": """Relying on proprietary systems controlled by a small number of companies creates a danger of concentrated power over the "information diet" of citizens. Open source enables diverse AI models that respect different cultures, values, and languages globally."""}
    },
    {
        "inputs": {"question": """How does LeCun suggest machines should perform reasoning and planning in the future?"""},
        "outputs": {"answer": """Future systems should perform planning through an optimization process in an abstract representation space. By using gradient-based inference to minimize an energy function, the model can deliberate on an answer before translating it into text."""}
    },
    {
        "inputs": {"question": """Is it possible to create an AI system that is completely unbiased?"""},
        "outputs": {"answer": """No, it is impossible. Bias is in the eye of the beholder, and different people have conflicting ideas about what constitutes bias. The solution is not to eliminate bias, but to ensure diversity of information sources, similar to a free press."""}
    }
]

In [ ]:
# create a dataset and examples in Langsmith
dataset_name = "eval-set-1"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples,
)

{'example_ids': ['f4cda954-38ac-46a4-905f-dbc75a4ae1c9',
  '6f34fbe7-9c40-4552-9ec7-9550b3740c4a',
  '3112aa3c-0adc-4f49-955a-169101fdc0a6',
  'ed3430f0-af6f-48d9-ae42-35339655b95f',
  '2192eaf6-ecc3-4633-a686-28f0ce72162a'],
 'count': 5,
 'as_of': '2026-08-04T23:39:01.495129065Z'}

### Evaluators

Correctness:
- Does the application generates the correct answer. 
    - Goal: Measure how similar or correct is the RAG answer relative to the ground truth
    - Mode: Requires a ground through (reference) answer supplied through evaluation dataset
    - Evaluator: Use LLM as judge to asses the correctness

In [13]:
from typing_extensions import Annotated, TypedDict

class CorrectnessGrade(TypedDict):
    explanation : Annotated[str, "Explain the reasoning for the score"]
    correct: Annotated[bool, "True if the answer is true otherwise False"]

# correctness prompt
correctness_instructions = """
You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and a STUDENT ANSWER.

Here is the grading criteria to follow:
1) Grade the student answer based ONLY on their factual accuracy relative to the ground through answer.
2) Ensure that the student answer does not contain any conflicting statement.
3) It is OK if the student answer contains more information than the ground answer, as long as it is factually accurate relative to the ground truth answer. 

Correctness: 
A correctness value of True means that the student's answer meets all the criteria.
A correctness value of False means that the student's answer does not meet all the criteria. 

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset.
""" 


In [14]:
from langchain_openai import ChatOpenAI

grader_llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0).with_structured_output(CorrectnessGrade, 
                                                                                      method = 'json_schema', 
                                                                                      strict = True)

# evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs:dict) -> bool:
    answers = f"""
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}
"""
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])

    return grade['correct']

Relevance: Response vs input
- Just check the inputs and output without looking at the reference_outputs. It will answer whether the model address the user's question or not

In [15]:
# grade output schema 
class RelevanceGrade(TypedDict):
    explaination: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the answer addresses the question. False if the answer fails to address the question"]

# relavance instructions
relevance_instructions = """
You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset.
"""

In [16]:
relevance_llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0).with_structured_output(RelevanceGrade, 
                                                                                      method = 'json_schema', 
                                                                                      strict = True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    answer = f"""
QUESTION: {inputs['question']}
STUDENT ANSWER: {outputs['answer']}
    """

    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer}
    ])

    return grade['relevant']

Groundedness: Response vs retrived docs
- How relevant is the response to the retrieved docs

In [17]:
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "True if the answer does not halucinates from the documents"]
# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""


In [18]:
grounded_llm = ChatOpenAI(model = "gpt-4o-mini", temperature =0).with_structured_output(GroundedGrade, 
                                                                                      method = 'json_schema', 
                                                                                      strict = True)

def groundedness(inputs: dict, outputs: dict) -> bool:
    retrived_text = "\n\n".join(outputs['relevant_chunks'])
    answer = f"FATCS:{retrived_text}\nSTUDENT ANSWER: {outputs['answer']}"

    grade = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer}
    ])

    return grade['grounded']

Retrieval Relevance: Retrieved docs vs input

In [19]:
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""


In [20]:
retrieval_relevance_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    retrived_text = "\n\n".join(outputs['relevant_chunks'])
    answer = f"FACTS: {retrived_text}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

## Run Evaluation

In [21]:
from langsmith import traceable

# define the function to run the full rag pipeline
@traceable()
def run_rag_pipeline(url, query):
    rag = RAGSearch(url = url)
    start = time.time()
    relevant_chunks = rag.search(query= query, top_k = 5)
    context = " ".join(relevant_chunks)
    response = rag.generate_response(context=context, query=query)
    timestamps = rag.get_video_timestamps()
    end = time.time()

    runtime_duration = end-start
    return {"answer": response, "relevant_chunks": relevant_chunks,"timestamps": timestamps, "runtime_duration": runtime_duration}

In [22]:
url = "https://www.youtube.com/watch?v=5t1vTLU7s40"

In [23]:
def target(inputs:dict) -> dict:
    return run_rag_pipeline(url = url, query=inputs['question'])

experiment_results = client.evaluate(
    target,
    data = dataset_name,
    evaluators = [correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gpt-4-0125-preview"}
)

View the evaluation results for experiment: 'rag-doc-relevance-49b23bed' at:
https://smith.langchain.com/o/92a32521-1999-4625-877a-20ef1a977765/datasets/50be5fbf-3a2c-4df5-91dc-aafeb589a124/compare?selectedSessions=d7576ef8-b2f6-439e-bfb3-869763da9849




5it [01:02, 12.57s/it]


In [27]:
eval_results = experiment_results.to_pandas()
eval_results

,inputs.question,outputs.answer,outputs.relevant_chunks,outputs.timestamps,outputs.runtime_duration,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,Is it possible to create an AI system that is ...,"Final Answer: No, it is absolutely not possibl...",[- The fundamental criticism that Gemini is ge...,"[(6448.62, 6562.71), (5704.938, 5851.94), (655...",3.775949,None,"No, it is impossible. Bias is in the eye of th...",True,True,True,True,3.861299,2192eaf6-ecc3-4633-a686-28f0ce72162a,019fcf25-e36e-7da2-9230-eee97a52815d
1,Why does LeCun argue that open-sourcing AI is ...,Final Answer: LeCun argues that open-sourcing ...,[because those systems will constitute the rep...,"[(5958.923, 6091.2), (5704.938, 5851.94), (872...",3.919534,None,Relying on proprietary systems controlled by a...,True,True,True,True,3.937690,3112aa3c-0adc-4f49-955a-169101fdc0a6,019fcf26-16e2-7ce0-a4cb-76e639596a3b
2,"What is a ""Joint-Embedding Predictive Architec...",Final Answer: A Joint-Embedding Predictive Arc...,"[Because you're like French, and ami is I gues...","[(1728.111, 1838.94), (1837.29, 1952.04), (523...",4.195482,None,Unlike generative models that spend resources ...,True,True,True,True,4.217263,6f34fbe7-9c40-4552-9ec7-9550b3740c4a,019fcf26-4af4-79d2-bbb1-6c8a6b9e012b
3,How does LeCun suggest machines should perform...,Final Answer: LeCun suggests that machines sho...,[or running a simulation or calling a calculat...,"[(9319.435, 9442.32), (125.82, 258.93), (9440....",4.127856,None,Future systems should perform planning through...,True,True,True,True,4.142805,ed3430f0-af6f-48d9-ae42-35339655b95f,019fcf26-7c5b-7652-a60c-4d2cbe5fbbca
4,Why does Yann LeCun believe auto-regressive LL...,Final Answer: Yann LeCun believes autoregressi...,[and fascinating discussions online as we do i...,"[(125.82, 258.93), (3268.98, 3377.76), (3021.2...",3.187677,None,LLMs lack four essential characteristics: an u...,True,True,True,True,3.197700,f4cda954-38ac-46a4-905f-dbc75a4ae1c9,019fcf26-aed8-72c1-b773-b7c2ba43a046


In [61]:
import pandas as pd
from datetime import datetime

def save_eval_results(obj:pd.DataFrame, dir_path:str = "eval_results"):
    """Save Evaluation results as excel"""
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    dir = Path(dir_path)
    dir.mkdir(exist_ok=True)
    file_path = dir / f"eval_{timestamp}.csv"
    obj.to_csv(file_path)

save_eval_results(obj=eval_results)

NameError: name 'eval_results' is not defined

### Programatic Eval
- Compare the timestamps of the correct answer with the timestamps from the most relevant chunks and calculate average variance

In [25]:
timestamp = "(1:30:00)"

In [3]:
def convert_to_seconds(timestamp:str) -> int:
    """Converts timestamp (eg. '00:02:30') from str into seconds in int"""
    timestamps = timestamp.lstrip("(").rstrip(")").split(":")
    seconds = 0
    for i, timestamp in enumerate(timestamps):
        if i == 0:      # hour hand
            seconds += int(timestamp) * 60 * 60    
        if i == 1:      # minute hand
            seconds += int(timestamp) * 60 
        if i == 2:      # seconds hand
            seconds += int(timestamp) * 1
    return seconds

In [63]:
convert_to_seconds(timestamp=timestamp)

5400

Create eval dataset

In [4]:
from langsmith import Client

client = Client()

dataset = client.create_dataset("RAG-eval-set-2")

In [5]:
import json
from pathlib import Path

file_path = Path("data/evals/eval_set_2.json")
with open(file_path, "rb") as f:
    data = json.load(f)

examples = data
examples[0]

{'question': 'What are the four essential characteristics of intelligent systems according to Yann LeCun?',
 'answer': 'Four essential characteristics: Capacity to understand the physical world, persistent memory, ability to reason, and ability to plan',
 'timestamp': '(0:02:49)'}

In [6]:
for example in examples:
    client.create_example(
        inputs = {"question": example["question"]},
        outputs= {"answer": example['answer']},
        metadata= {"timestamp": convert_to_seconds(example['timestamp'])},
        dataset_id= dataset.id
    )

## Evaluators

Correctness Grade

In [7]:
from typing_extensions import Annotated, TypedDict

class CorrectnessGrade(TypedDict):
    grade: Annotated[int, "Grade from 1-10"]
    correct: Annotated[bool, "True if the answer is true otherwise False"]
    explaination : Annotated[str, "Explain the reasoning for the score"]

# correctness prompt
correctness_instructions = """
You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and a STUDENT ANSWER.

Your job is to evaluate the student answer in two ways:
1. Assign a SCORE from 1 to 10.
2. Assign a CORRECTNESS value of True or False.


- SCORE 1 means the answer is very poor: mostly or entirely incorrect, unrelated to the ground truth, or highly misleading.
- SCORE 10 means the answer is excellent: fully correct, closely aligned with the ground truth, and clearly expressed.

Here is the grading criteria to follow:

1) Grade the student answer based ONLY on its factual accuracy relative to the ground truth answer.
2) Ensure that the student answer does not contain any conflicting or contradictory statements compared to the ground truth.
3) It is OK if the student answer contains more information than the ground truth answer, as long as all additional information is factually accurate and consistent with the ground truth.
4) If the student answer includes partially correct information but also some incorrect or conflicting statements, deduct marks accordingly. The more serious or numerous the conflicts, the lower the score should be.
5) If the student answer is vague, incomplete, or only loosely related to the ground truth, give a mid-to-low score depending on how much correct information it contains.

Scoring guidelines:
- 9–10: Fully correct, no conflicts, closely matches or appropriately extends the ground truth.
- 7–8: Mostly correct, minor omissions or minor issues, no serious conflicts.
- 5–6: Partially correct, noticeable gaps or mild conflicts, but still shows some understanding.
- 3–4: Mostly incorrect or poorly aligned, with limited correct information.
- 1–2: Very poor, largely or entirely incorrect, unrelated, or highly conflicting with the ground truth.

Output format:
Provide your assement in the form of JSON object with following fields:
{
    "grade": <integer between 1 to 10>,
    "correct": <True or False>,
    "explaination": "<Explain your reasoning step by step manner, referencing specific parts of the student answer and the ground truth.>"
}

Avoid simply stating the correct answer at the outset. Focus on comparing the student answer to the ground truth and justifying the score.
""" 

In [8]:
from langchain_openai import ChatOpenAI

grader_llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0).with_structured_output(CorrectnessGrade, 
                                                                                      method = 'json_schema', 
                                                                                      strict = True)

# evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs:dict) -> bool:
    answers = f"""
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}
"""
    result = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])

    return {
        "key": "correctness",
        "score": result["grade"],
        "value": result["correct"],
        "comment": result["explaination"]
    }   

Relevance: Response vs input

In [9]:
class RelevanceGrade(TypedDict):
    grade: Annotated[int, "Grade from 1-10"]
    relevant: Annotated[bool, "True if the answer addresses the question. False if the answer fails to address the question"]
    explaination : Annotated[str, "Explain the reasoning for the score"]

relevance_instructions = """
You are a teacher grading a quiz.

You will be given a QUESTION and a STUDENT ANSWER.

Your job is to evaluate the student answer in two ways:
1. Assign a SCORE from 1 to 10 for relevance.
2. Assign a RELEVANCE value of True or False.

- SCORE 1 means the answer is very poor: mostly or entirely irrelevant, off-topic, or unhelpful for answering the question.
- SCORE 10 means the answer is excellent: highly relevant, directly addresses the question, and is clearly expressed.

Here is the grading criteria to follow:

1) Evaluate how directly the STUDENT ANSWER addresses the QUESTION.
2) Ensure the STUDENT ANSWER is concise and focused on the QUESTION, without unnecessary digressions.
3) The STUDENT ANSWER should meaningfully help to answer the QUESTION (not just restate it or talk around it).
4) If the STUDENT ANSWER is partially relevant but includes off-topic or distracting content, deduct marks accordingly.
5) If the STUDENT ANSWER is vague, generic, or only loosely connected to the QUESTION, give a mid-to-low score depending on how much it actually helps answer the QUESTION.

Scoring guidelines:
- 9–10: Highly relevant, directly answers the question, clear and focused.
- 7–8: Mostly relevant, minor digressions or slight lack of focus, but still clearly helps answer the question.
- 5–6: Partially relevant, noticeable vagueness or off-topic content, but some helpful information.
- 3–4: Mostly irrelevant or unhelpful, with limited connection to the question.
- 1–2: Very poor, largely or entirely irrelevant, off-topic, or confusing.

Relevance:
- Relevance = True if the answer is substantially relevant and clearly helps answer the QUESTION (typically scoring 7 or above).
- Relevance = False if the answer is mostly irrelevant, unhelpful, or only weakly connected to the QUESTION (typically scoring 6 or below).

Output format:
Provide your assessment in the form of a JSON object with the following fields:

{
  "grade": <integer between 1 and 10>,
  "relevant": <True or False>,
  "explaination": "<Explain your reasoning step by step, referencing specific parts of the student answer and the question.>"
}

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset. Focus on how well the student answer addresses the QUESTION and justifying the score.
"""

In [10]:
relevance_llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0).with_structured_output(RelevanceGrade, 
                                                                                      method = 'json_schema', 
                                                                                      strict = True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    answer = f"""
QUESTION: {inputs['question']}
STUDENT ANSWER: {outputs['answer']}
    """

    result = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer}
    ])

    return {
        "key": "correctness",
        "score": result["grade"],
        "value": result["relevant"],
        "comment": result["explaination"]
    } 

Groundedness: Response vs retrived docs

In [11]:
class GroundedGrade(TypedDict):
    grade: Annotated[int, "Grade from 1-10"]
    grounded: Annotated[bool, "True if the answer does not halucinates from the documents. False if the answer is made up"]
    explaination : Annotated[str, "Explain the reasoning for the score"]

grounded_instructions = """
You are a teacher grading a quiz.

You will be given FACTS and a STUDENT ANSWER.

Your job is to evaluate the student answer in two ways:
1. Assign a SCORE from 1 to 10 for groundedness.
2. Assign a GROUNDED value of True or False.

- SCORE 1 means the answer is very poor: mostly or entirely unsupported by the FACTS, highly hallucinated, or misleading.
- SCORE 10 means the answer is excellent: fully supported by the FACTS, with no hallucinated information, and clearly expressed.

Here is the grading criteria to follow:

1) Evaluate whether the STUDENT ANSWER is fully grounded in the FACTS provided.
2) The STUDENT ANSWER should not introduce any information that is not supported by, implied by, or consistent with the FACTS.
3) Additional details are acceptable only if they are clearly supported by the FACTS and do not contradict them.
4) If the STUDENT ANSWER contains partially grounded information but also some unsupported or hallucinated claims, deduct marks accordingly. The more serious or numerous the unsupported claims, the lower the score should be.
5) If the STUDENT ANSWER is vague, speculative, or goes beyond the FACTS in a way that cannot be justified by them, give a mid-to-low score depending on how much of the answer is actually grounded.

Scoring guidelines:
- 9–10: Fully grounded, no hallucinations, all claims supported by or clearly implied by the FACTS.
- 7–8: Mostly grounded, minor speculative or unclear elements, but no serious unsupported claims.
- 5–6: Partially grounded, noticeable unsupported or speculative content, but some alignment with the FACTS.
- 3–4: Mostly ungrounded, with limited connection to the FACTS and several unsupported claims.
- 1–2: Very poor, largely or entirely hallucinated, unsupported by the FACTS, or contradictory to them.

Grounded:
- Grounded = True if the answer is substantially supported by the FACTS, with no major hallucinations (typically scoring 7 or above).
- Grounded = False if the answer contains significant hallucinated, unsupported, or contradictory information (typically scoring 6 or below).

Output format:
Provide your assessment in the form of a JSON object with the following fields:

{
  "grade": <integer between 1 and 10>,
  "grounded": <True or False>,
  "explaination": "<Explain your reasoning step by step, referencing specific parts of the student answer and the FACTS.>"
}

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset. Focus on how well the student answer is grounded in the FACTS and justifying the score.

"""

In [12]:
grounded_llm = ChatOpenAI(model = "gpt-4o-mini", temperature =0).with_structured_output(GroundedGrade, 
                                                                                      method = 'json_schema', 
                                                                                      strict = True)

def groundedness(inputs: dict, outputs: dict) -> bool:
    retrived_text = "\n\n".join(outputs['relevant_chunks'])
    answer = f"FATCS:{retrived_text}\nSTUDENT ANSWER: {outputs['answer']}"

    result = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer}
    ])

    return {
        "key": "correctness",
        "score": result["grade"],
        "value": result["grounded"],
        "comment": result["explaination"]
    } 

Retrieval Relevance: Retrieved docs vs input

In [13]:
class RetrievalRelevanceGrade(TypedDict):
    grade: Annotated[int, "Grade from 1-10"]
    relevant: Annotated[bool, "True if the retrieved documents are relevant to the question, False otherwise"]
    explaination : Annotated[str, "Explain the reasoning for the score"]

retrieval_relevance_instructions = """
You are a teacher grading a quiz.

You will be given a QUESTION and a set of FACTS (retrieved documents) provided by the student.

Your job is to evaluate the relevance of the FACTS to the QUESTION in two ways:
1. Assign a SCORE from 1 to 10 for relevance.
2. Assign a RELEVANCE value of True or False.

- SCORE 1 means the FACTS are very poor: completely or almost completely unrelated to the QUESTION.
- SCORE 10 means the FACTS are excellent: clearly and strongly related to the QUESTION, with substantial semantic overlap.

Here is the grading criteria to follow:

1) Your goal is to identify whether the FACTS are related to the QUESTION.
2) If the FACTS contain ANY keywords, concepts, or semantic meaning related to the QUESTION, consider them relevant to some degree.
3) It is OK if the FACTS contain SOME information that is unrelated to the QUESTION, as long as they also contain information that is clearly related.
4) If only a small portion of the FACTS is related and most of the content is off-topic, give a mid-to-low score depending on how much relevant information is present.
5) If the FACTS are entirely off-topic, generic, or about a different subject, give a very low score.

Scoring guidelines:
- 9–10: Highly relevant, strong semantic overlap with the QUESTION, clearly useful for answering it.
- 7–8: Mostly relevant, good overlap with the QUESTION, some off-topic content but still clearly useful.
- 5–6: Partially relevant, noticeable off-topic content, but some meaningful connection to the QUESTION.
- 3–4: Weakly relevant, only minor or superficial connection to the QUESTION.
- 1–2: Very poor, largely or completely unrelated to the QUESTION.

Relevance:
- Relevance = True if the FACTS contain ANY meaningful keywords, concepts, or semantic content related to the QUESTION (typically scoring 5 or above).
- Relevance = False if the FACTS are completely or almost completely unrelated to the QUESTION (typically scoring 4 or below).

Output format:
Provide your assessment in the form of a JSON object with the following fields:

{
  "grade": <integer between 1 and 10>,
  "relevant": <True or False>,
  "explaination": "<Explain your reasoning step by step, referencing specific parts of the FACTS and the QUESTION.>"
}

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset. Focus on how well the FACTS relate to the QUESTION and justifying the score.
"""

In [14]:
retrieval_relevance_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    retrived_text = "\n\n".join(outputs['relevant_chunks'])
    answer = f"FACTS: {retrived_text}\nQUESTION: {inputs['question']}"

    # Run evaluator
    result = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return {
        "key": "correctness",
        "score": result["grade"],
        "value": result["relevant"],
        "comment": result["explaination"]
    } 

Timestamp Error
- How far off retrieval is from the true timestamp of the answer. 
- Will only consider the timestamp of the most relevant chunk

In [15]:
def timestamp_error(outputs: dict, reference_outputs:dict) -> dict:
    correct_ts = reference_outputs['timestamp']
    retrieved_ts = outputs['timestamp']

    error = abs(correct_ts - retrieved_ts)

    return {
        "key": "timestamp_error",
        "score": error,
        "value": error,  # optional, but allowed
        "comment": f"The retrieved document timestamp ({retrieved_ts}s) differs from the correct timestamp ({correct_ts}s) by {error} seconds."
    }


### Run Evals

In [16]:
from langsmith import traceable

# define the function to run the full rag pipeline
@traceable()
def run_rag_pipeline(url, query):
    rag = RAGSearch(url = url)
    start = time.time()
    relevant_chunks = rag.search(query= query, top_k = 5)
    context = " ".join(relevant_chunks)
    response = rag.generate_response(context=context, query=query)
    timestamp = rag.get_video_timestamps()[0][0]
    end = time.time()

    runtime_duration = end-start
    return {"answer": response, "relevant_chunks": relevant_chunks,"timestamp": timestamp, "runtime_duration": runtime_duration}

In [17]:
url = "https://www.youtube.com/watch?v=5t1vTLU7s40"

In [ ]:
def target(inputs:dict) -> dict:
    return run_rag_pipeline(url = url, query=inputs['question'])

experiment_results = client.evaluate(
    target,
    data = "RAG-eval-set-2",
    evaluators = [correctness, groundedness, relevance, retrieval_relevance, timestamp_error],
    experiment_prefix="rag-detail-evaluation",
    metadata={"version": "LCEL context, gpt-4-0125-preview"}
)

View the evaluation results for experiment: 'rag-detail-evaluation-ee6cadf4' at:
https://smith.langchain.com/o/92a32521-1999-4625-877a-20ef1a977765/datasets/c356e917-8be5-43d4-a925-5f59b5a36240/compare?selectedSessions=3ff934bd-bc4b-4a8b-9e24-297e8049ebd2




16it [04:38, 14.73s/it]